# Corrected Preprocessing Pipeline

## Keputusan preprocessing

- `X` dan `y` dipisahkan sebelum pembagian data. `target_binary` menjadi `y`; `target` dan `source_file` dikeluarkan dari predictor untuk mencegah leakage.
- Data dibagi menggunakan stratified train-test split. Stratifikasi menjaga proporsi kelas pada train dan test.
- Semua transformer di dalam pipeline hanya di-fit pada training set. Dengan demikian median imputasi, kategori hasil encoding, dan parameter scaling tidak melihat test set.
- Missing values numerik diisi dengan median training, sedangkan missing values kategorikal diisi dengan modus training.
- Logistic Regression sensitif terhadap skala fitur numerik, sehingga `StandardScaler` digunakan pada fitur numerik. One-hot encoded features tidak diskalakan lagi.
- Tidak digunakan SMOTE, feature selection tambahan, maupun hyperparameter tuning. Model dan parameternya ditetapkan satu kali.
- Test set hanya digunakan untuk evaluasi final setelah pipeline selesai di-fit pada training set.

In [4]:
# fetch data
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

DATA_DIR = Path('.\\heart+disease')
FILE_NAMES = [
    "processed.cleveland.data",
    "processed.hungarian.data",
    "processed.switzerland.data",
    "processed.va.data",
]
FEATURE_NAMES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"
]

frames = []
for file_name in FILE_NAMES:
    file_path = DATA_DIR / file_name
    if not file_path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {file_path}")
    frame = pd.read_csv(file_path, header=None, names=FEATURE_NAMES, na_values="?")
    frame["source_file"] = file_name
    frames.append(frame)

df = pd.concat(frames, ignore_index=True)
df[FEATURE_NAMES] = df[FEATURE_NAMES].apply(pd.to_numeric, errors="coerce")
df["target_binary"] = (df["target"] > 0).astype("Int64")
df["target_binary"] = df["target_binary"].astype(str)

print(f"Dataset berhasil dimuat: {df.shape[0]:,} observasi, {len(FEATURE_NAMES):,} fitur asli")
display(df.head())

Dataset berhasil dimuat: 920 observasi, 14 fitur asli


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target,source_file,target_binary
0,63.000,1.000,1.000,145.000,233.000,1.000,2.000,150.000,0.000,2.300,3.000,0.000,6.000,0,processed.cleveland.data,0
1,67.000,1.000,4.000,160.000,286.000,0.000,2.000,108.000,1.000,1.500,2.000,3.000,3.000,2,processed.cleveland.data,1
2,67.000,1.000,4.000,120.000,229.000,0.000,2.000,129.000,1.000,2.600,2.000,2.000,7.000,1,processed.cleveland.data,1
3,37.000,1.000,3.000,130.000,250.000,0.000,0.000,187.000,0.000,3.500,3.000,0.000,3.000,0,processed.cleveland.data,0
4,41.000,0.000,2.000,130.000,204.000,0.000,2.000,172.000,0.000,1.400,1.000,0.000,3.000,0,processed.cleveland.data,0


In [5]:
# ubah target column to target_binary
df.drop(columns=["target"], inplace=True)
df.rename(columns={"target_binary": "target"}, inplace=True)
df

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,source_file,target
0,63.000,1.000,1.000,145.000,233.000,1.000,2.000,150.000,0.000,2.300,3.000,0.000,6.000,processed.cleveland.data,0
1,67.000,1.000,4.000,160.000,286.000,0.000,2.000,108.000,1.000,1.500,2.000,3.000,3.000,processed.cleveland.data,1
2,67.000,1.000,4.000,120.000,229.000,0.000,2.000,129.000,1.000,2.600,2.000,2.000,7.000,processed.cleveland.data,1
3,37.000,1.000,3.000,130.000,250.000,0.000,0.000,187.000,0.000,3.500,3.000,0.000,3.000,processed.cleveland.data,0
4,41.000,0.000,2.000,130.000,204.000,0.000,2.000,172.000,0.000,1.400,1.000,0.000,3.000,processed.cleveland.data,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
915,54.000,0.000,4.000,127.000,333.000,1.000,1.000,154.000,0.000,0.000,NaN,NaN,NaN,processed.va.data,1
916,62.000,1.000,1.000,NaN,139.000,0.000,1.000,NaN,NaN,NaN,NaN,NaN,NaN,processed.va.data,0
917,55.000,1.000,4.000,122.000,223.000,1.000,1.000,100.000,0.000,0.000,NaN,NaN,6.000,processed.va.data,1
918,58.000,1.000,4.000,NaN,385.000,1.000,2.000,NaN,NaN,NaN,NaN,NaN,NaN,processed.va.data,0


In [6]:
# change datatype for categorical column
columns = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal', 'target']
for col in columns:
    df[col] = df[col].astype('category')
df

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,source_file,target
0,63.000,1.000,1.000,145.000,233.000,1.000,2.000,150.000,0.000,2.300,3.000,0.000,6.000,processed.cleveland.data,0
1,67.000,1.000,4.000,160.000,286.000,0.000,2.000,108.000,1.000,1.500,2.000,3.000,3.000,processed.cleveland.data,1
2,67.000,1.000,4.000,120.000,229.000,0.000,2.000,129.000,1.000,2.600,2.000,2.000,7.000,processed.cleveland.data,1
3,37.000,1.000,3.000,130.000,250.000,0.000,0.000,187.000,0.000,3.500,3.000,0.000,3.000,processed.cleveland.data,0
4,41.000,0.000,2.000,130.000,204.000,0.000,2.000,172.000,0.000,1.400,1.000,0.000,3.000,processed.cleveland.data,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
915,54.000,0.000,4.000,127.000,333.000,1.000,1.000,154.000,0.000,0.000,NaN,NaN,NaN,processed.va.data,1
916,62.000,1.000,1.000,NaN,139.000,0.000,1.000,NaN,NaN,NaN,NaN,NaN,NaN,processed.va.data,0
917,55.000,1.000,4.000,122.000,223.000,1.000,1.000,100.000,0.000,0.000,NaN,NaN,6.000,processed.va.data,1
918,58.000,1.000,4.000,NaN,385.000,1.000,2.000,NaN,NaN,NaN,NaN,NaN,NaN,processed.va.data,0


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   age          920 non-null    float64 
 1   sex          920 non-null    category
 2   cp           920 non-null    category
 3   trestbps     861 non-null    float64 
 4   chol         890 non-null    float64 
 5   fbs          830 non-null    category
 6   restecg      918 non-null    category
 7   thalach      865 non-null    float64 
 8   exang        865 non-null    category
 9   oldpeak      858 non-null    float64 
 10  slope        611 non-null    category
 11  ca           309 non-null    category
 12  thal         434 non-null    category
 13  source_file  920 non-null    str     
 14  target       920 non-null    category
dtypes: category(9), float64(5), str(1)
memory usage: 72.3 KB


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Pisahkan predictor dan target sebelum split.
# target_binary adalah label; target asli dan source_file tidak boleh masuk ke X.
X = df.drop(columns=["target", "source_file"]).copy()
y = df["target"].astype(int).copy()

print(f"X shape sebelum split: {X.shape}")
print(f"y shape sebelum split: {y.shape}")
print("Distribusi y:")
display(y.value_counts(normalize=True).rename("proportion").to_frame())

# 2. Split dilakukan sebelum transformer dibuat dan di-fit.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Proporsi kelas train: {y_train.mean():.3f}")
print(f"Proporsi kelas test: {y_test.mean():.3f}")

# Kolom kategorikal diperlakukan sebagai kategori walaupun tersimpan sebagai angka.
numeric_features_model = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categorical_features_model = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features_model),
        ("categorical", categorical_pipeline, categorical_features_model),
    ],
    remainder="drop",
)

# 3. Pipeline mem-fit preprocessing dan model hanya menggunakan training set.
# Tidak ada SMOTE, feature selection, atau tuning tambahan.
model_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

model_pipeline.fit(X_train, y_train)
print("Pipeline berhasil di-fit pada training set.")

# Verifikasi bahwa parameter imputer berasal dari training set dan test belum di-fit.
trained_preprocessor = model_pipeline.named_steps["preprocessing"]
trained_numeric_imputer = trained_preprocessor.named_transformers_["numeric"].named_steps["imputer"]
print("Median imputasi training:")
display(pd.Series(trained_numeric_imputer.statistics_, index=numeric_features_model, name="training_median"))

X shape sebelum split: (920, 13)
y shape sebelum split: (920,)
Distribusi y:


,proportion
target,
1,0.553
0,0.447


X_train: (736, 13), X_test: (184, 13)
Proporsi kelas train: 0.553
Proporsi kelas test: 0.554
Pipeline berhasil di-fit pada training set.
Median imputasi training:


age         54.000
trestbps   130.000
chol       223.000
thalach    140.000
oldpeak      0.500
Name: training_median, dtype: float64

## Final evaluation

Prediksi pada `X_test` dilakukan hanya setelah seluruh preprocessing dan model selesai di-fit dengan `X_train`. Tidak ada informasi dari test set yang digunakan untuk memilih atau mengubah parameter pipeline.

In [10]:
# 4. Test set hanya dipakai untuk final evaluation.
y_test_pred = model_pipeline.predict(X_test)
y_test_proba = model_pipeline.predict_proba(X_test)[:, 1]

print("FINAL TEST EVALUATION")
print(f"Accuracy : {accuracy_score(y_test, y_test_pred):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_test_proba):.3f}")
print("\nClassification report:")
print(classification_report(y_test, y_test_pred, target_names=["No disease", "Disease"]))

confusion = pd.DataFrame(
    confusion_matrix(y_test, y_test_pred),
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"],
)
display(confusion)

print("Jumlah fitur setelah preprocessing:", len(model_pipeline.named_steps["preprocessing"].get_feature_names_out()))

FINAL TEST EVALUATION
Accuracy : 0.842
ROC-AUC  : 0.909

Classification report:
              precision    recall  f1-score   support

  No disease       0.84      0.80      0.82        82
     Disease       0.85      0.87      0.86       102

    accuracy                           0.84       184
   macro avg       0.84      0.84      0.84       184
weighted avg       0.84      0.84      0.84       184



,Predicted 0,Predicted 1
Actual 0,66,16
Actual 1,13,89


Jumlah fitur setelah preprocessing: 28
